# Polymer Property Prediction — Reproducible Kaggle Notebook

**Targets:** `tg` (Glass Transition Temperature) and `egc` (Chain Bandgap), predicted from polymer SMILES.
**Metric:** mean R² across both target types.
**Models:** LightGBM + XGBoost + CatBoost per target, 5-fold GroupKFold (canonical-SMILES grouped), OOF-only Ridge stacking, leakage-free exact-match override.

This notebook is fully self-contained: every cell below runs top-to-bottom with no manual edits and produces `submission.csv`. Random seeds are fixed throughout (`SEED = 42`) for reproducibility across runs.

## SECTION 1 — CONFIGURATION

Global seeds, determinism settings, GPU detection, and library version logging. Everything downstream depends on `SEED = 42` being fixed before any model or split is created.

In [ ]:
import os, sys, random, json, time, hashlib, pickle, warnings
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np
random.seed(SEED)
np.random.seed(SEED)

N_FOLDS = 5
MAX_ESTIMATORS = 3000          # Kaggle-scale budget; early stopping typically halts well before this
EARLY_STOPPING_ROUNDS = 150
EARLY_STOP_HOLDOUT_FRAC = 0.15

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def detect_gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

GPU_AVAILABLE = detect_gpu()
LGB_PARAMS_EXTRA = {}
XGB_TREE_METHOD = "hist"
CB_TASK_TYPE = "CPU"  # CPU-only for bit-reproducible results across runs (CatBoost GPU is nondeterministic)

print("="*70)
print("SECTION 1 — CONFIGURATION")
print("="*70)
print(f"SEED = {SEED}")
print(f"N_FOLDS = {N_FOLDS}")
print(f"GPU available: {GPU_AVAILABLE}")
print(f"numpy version: {np.__version__}")
print(f"lightgbm version: {lgb.__version__}")
print(f"xgboost version: {xgb.__version__}")
print(f"catboost version: {cb.__version__}")


## SECTION 2 — IMPORTS

All imports used by this notebook, and nothing else — no dead imports.

In [ ]:
import subprocess, sys, os as _os
try:
    import rdkit
except ImportError:
    print('rdkit not found - installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'rdkit'])
    import rdkit
print('rdkit version:', rdkit.__version__)
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

## SECTION 3 — DATA LOADING

Loads `train.csv`, `test.csv`, and `sample_submission.csv` from whichever Kaggle input directory actually contains them (auto-detected), and prints shape/columns/memory/missing-value diagnostics for each.

In [ ]:
print("\n" + "="*70)
print("SECTION 3 — DATA LOADING")
print("="*70)

def resolve_data_dir():
    roots = []
    if os.path.isdir("/kaggle/input"):
        roots.append("/kaggle/input")
    roots += ["/mnt/user-data/uploads", "."]
    print("[resolve_data_dir] roots:", roots)
    seen = set()
    for r in roots:
        if not os.path.isdir(r):
            continue
        stack = [r]
        while stack:
            cur = stack.pop()
            try:
                entries = os.listdir(cur)
            except Exception:
                continue
            has_tr = os.path.exists(os.path.join(cur, "train.csv"))
            has_te = os.path.exists(os.path.join(cur, "test.csv"))
            if has_tr and has_te:
                print("[resolve_data_dir] found data dir:", cur)
                return cur
            for e in entries:
                p = os.path.join(cur, e)
                if os.path.isdir(p) and p not in seen:
                    seen.add(p)
                    stack.append(p)
    print("[resolve_data_dir] exhaustive search complete - train/test not found under any root.")
    raise FileNotFoundError("Could not locate train.csv/test.csv in any known input directory.")

DATA_DIR = resolve_data_dir()
print(f"Using data directory: {DATA_DIR}")

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path) if os.path.exists(sample_sub_path) else None

for name, df in [("train", train), ("test", test)]:
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 3))
    print("missing values per column:\n", df.isna().sum().to_dict())

if sample_sub is not None:
    print("\n--- sample_submission ---")
    print("shape:", sample_sub.shape, "columns:", list(sample_sub.columns))

## SECTION 4 — DATA VALIDATION

Automated checks for invalid SMILES, missing targets, raw and canonical-SMILES duplicates, per-target_type sample imbalance, and train/test molecular overlap (both exact-string and canonical-structure level).

In [ ]:
print("\n" + "="*70)
print("SECTION 4 — DATA VALIDATION")
print("="*70)

def canonicalize(smi):
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m, canonical=True) if m is not None else None

train_invalid = train['smiles'].apply(lambda s: Chem.MolFromSmiles(s) is None).sum()
test_invalid = test['smiles'].apply(lambda s: Chem.MolFromSmiles(s) is None).sum()
print(f"Invalid/unparseable SMILES -> train: {train_invalid}, test: {test_invalid}")
if train_invalid > 0 or test_invalid > 0:
    raise ValueError("Invalid SMILES detected — cannot proceed safely. Inspect and clean before rerunning.")

print(f"Missing target values in train: {train['target'].isna().sum()}")
print(f"Raw duplicate SMILES rows in train: {train['smiles'].duplicated().sum()}")

train['canonical_smiles'] = train['smiles'].apply(canonicalize)
test['canonical_smiles'] = test['smiles'].apply(canonicalize)

print(f"Duplicate canonical SMILES rows in train: {train['canonical_smiles'].duplicated().sum()}")

print("\nTarget type distribution (train):")
print(train['target_type'].value_counts())
print("\nTarget type distribution (test):")
print(test['target_type'].value_counts())

imbalance_ratio = train['target_type'].value_counts(normalize=True)
print("\nTarget imbalance (fraction of rows per target_type):")
print(imbalance_ratio)

exact_overlap = set(train['smiles']) & set(test['smiles'])
canon_overlap = set(train['canonical_smiles']) & set(test['canonical_smiles'])
print(f"\nExact SMILES overlap train/test: {len(exact_overlap)} molecules")
print(f"Canonical SMILES overlap train/test: {len(canon_overlap)} molecules")
if len(canon_overlap) > 0:
    print("-> These molecules will be forced into the SAME CV fold as their test-set "
          "twin's group to prevent leakage (handled via GroupKFold on canonical SMILES in Section 6).")

within_target_dupe_groups = {}
for tt in train['target_type'].unique():
    sub = train[train['target_type'] == tt]
    n_dupe = sub['canonical_smiles'].duplicated().sum()
    within_target_dupe_groups[tt] = int(n_dupe)
print(f"\nDuplicate canonical-SMILES rows within each target_type (train): {within_target_dupe_groups}")

## SECTION 5 — FEATURE ENGINEERING

Full RDKit descriptor set (~208 physicochemical descriptors), a 256-bit Morgan count fingerprint, 167-bit MACCS keys, and two graph-theoretic features (Wiener index, ring count) for every unique molecule across train ∪ test. Computed once and cached to disk (`feature_cache/`, keyed by a hash of the SMILES set) so reruns skip recomputation entirely.

In [ ]:
print("\n" + "="*70)
print("SECTION 5 — FEATURE ENGINEERING")
print("="*70)

all_smiles = pd.concat([train['smiles'], test['smiles']]).unique()
print(f"Unique SMILES to featurize (train ∪ test): {len(all_smiles)}")

smiles_hash = hashlib.md5("".join(sorted(all_smiles)).encode()).hexdigest()[:12]
cache_dir = os.path.join(os.getcwd(), "feature_cache")
os.makedirs(cache_dir, exist_ok=True)
cache_path = os.path.join(cache_dir, f"features_{smiles_hash}.pkl")

MORGAN_BITS = 256
mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MORGAN_BITS)

def compute_features(smi):
    m = Chem.MolFromSmiles(smi)
    desc = Descriptors.CalcMolDescriptors(m)
    morgan = list(mfgen.GetCountFingerprintAsNumPy(m))
    maccs = list(MACCSkeys.GenMACCSKeys(m))
    dist_matrix = Chem.GetDistanceMatrix(m)
    wiener_index = float(dist_matrix.sum()) / 2.0
    num_rings = m.GetRingInfo().NumRings()
    return desc, morgan, maccs, wiener_index, num_rings

if os.path.exists(cache_path):
    print(f"Loading cached features from {cache_path}")
    with open(cache_path, "rb") as f:
        feature_rows = pickle.load(f)
else:
    print("Computing descriptors (RDKit full descriptor set, Morgan count fingerprint, "
          "MACCS keys, Wiener index, ring count) — cached to disk for reuse on rerun.")
    t0 = time.time()
    feature_rows = []
    for smi in all_smiles:
        desc, morgan, maccs, wiener_index, num_rings = compute_features(smi)
        feature_rows.append((smi, desc, morgan, maccs, wiener_index, num_rings))
    print(f"Feature computation time: {round(time.time()-t0, 1)}s for {len(all_smiles)} molecules")
    with open(cache_path, "wb") as f:
        pickle.dump(feature_rows, f)

desc_df = pd.DataFrame([r[1] for r in feature_rows])
desc_df.columns = [f"rdkit_{c}" for c in desc_df.columns]
morgan_df = pd.DataFrame([r[2] for r in feature_rows], columns=[f"morgan_{i}" for i in range(MORGAN_BITS)])
maccs_df = pd.DataFrame([r[3] for r in feature_rows], columns=[f"maccs_{i}" for i in range(167)])
graph_df = pd.DataFrame({
    "graph_wiener_index": [r[4] for r in feature_rows],
    "graph_num_rings": [r[5] for r in feature_rows],
})
smiles_col = pd.DataFrame({"smiles": [r[0] for r in feature_rows]})

feature_table = pd.concat([smiles_col, desc_df, morgan_df, maccs_df, graph_df], axis=1)
feature_cols = [c for c in feature_table.columns if c != "smiles"]

feature_table[feature_cols] = feature_table[feature_cols].replace([np.inf, -np.inf], np.nan)
col_means = feature_table[feature_cols].mean()
feature_table[feature_cols] = feature_table[feature_cols].fillna(col_means).fillna(0.0)
feature_table[feature_cols] = feature_table[feature_cols].clip(-1e10, 1e10)

print(f"Final feature matrix: {feature_table.shape[0]} molecules x {len(feature_cols)} features")

train = train.merge(feature_table, on="smiles", how="left")
test = test.merge(feature_table, on="smiles", how="left")
assert train[feature_cols].isna().sum().sum() == 0, "Unexpected NaNs after feature merge (train)"
assert test[feature_cols].isna().sum().sum() == 0, "Unexpected NaNs after feature merge (test)"

## SECTION 6 — CROSS VALIDATION STRATEGY

Automatically selects GroupKFold (grouped on canonical SMILES) per target_type based on the duplicate-detection results from Section 4, and prints the reasoning for the chosen scheme.

In [ ]:
print("\n" + "="*70)
print("SECTION 6 — CROSS VALIDATION STRATEGY")
print("="*70)

cv_strategy = {}
for tt in train['target_type'].unique():
    sub = train[train['target_type'] == tt]
    n_dupe = int(sub['canonical_smiles'].duplicated().sum())
    if n_dupe > 0:
        cv_strategy[tt] = "GroupKFold"
        print(f"[{tt}] {n_dupe} duplicate canonical-SMILES groups detected -> GroupKFold on "
              f"canonical_smiles chosen to keep duplicate molecules on the same side of every fold "
              f"(prevents identical-molecule leakage between train and validation).")
    else:
        cv_strategy[tt] = "GroupKFold"
        print(f"[{tt}] No exact duplicate canonical SMILES found, but GroupKFold on canonical_smiles "
              f"is used uniformly across all targets as a conservative, leakage-safe default "
              f"(protects against any duplicates introduced by future data updates).")

## SECTION 7 — MODEL TRAINING (per target_type: LightGBM, CatBoost, XGBoost)

Trains LightGBM, XGBoost, and CatBoost independently for each target_type using 5-fold GroupKFold. Early stopping is computed on a holdout carved OUT OF the training fold only (never the validation fold), so the early-stopping criterion cannot leak into the OOF score. Prints per-fold R², and mean/std per model.

In [ ]:
print("\n" + "="*70)
print("SECTION 7 \u2014 MODEL TRAINING")
print("="*70)

def make_models():
    lgb_model = lgb.LGBMRegressor(
        n_estimators=MAX_ESTIMATORS, learning_rate=0.03, num_leaves=15,
        min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbose=-1
    )
    xgb_model = xgb.XGBRegressor(
        n_estimators=MAX_ESTIMATORS, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, tree_method=XGB_TREE_METHOD,
        random_state=SEED, verbosity=0, early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )
    cb_model = cb.CatBoostRegressor(
        iterations=MAX_ESTIMATORS, learning_rate=0.03, depth=6,
        random_seed=SEED, task_type=CB_TASK_TYPE, verbose=False,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )
    return lgb_model, xgb_model, cb_model

results = {}
oof_store = {}
test_pred_store = {}
fold_importance_store = {}

for tt in sorted(train['target_type'].unique()):
    print(f"\n----- Target: {tt} -----")
    tr = train[train['target_type'] == tt].reset_index(drop=True)
    te = test[test['target_type'] == tt].reset_index(drop=True)

    X = tr[feature_cols].values
    y = tr['target'].values
    groups = tr['canonical_smiles'].values
    X_test = te[feature_cols].values

    gkf = GroupKFold(n_splits=N_FOLDS)
    n = len(y)
    oof_lgb, oof_xgb, oof_cb = np.zeros(n), np.zeros(n), np.zeros(n)
    test_pred_lgb = np.zeros((N_FOLDS, len(te)))
    test_pred_xgb = np.zeros((N_FOLDS, len(te)))
    test_pred_cb = np.zeros((N_FOLDS, len(te)))
    fold_scores = {"lgb": [], "xgb": [], "cb": []}
    importances = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr_full, X_va = X[tr_idx], X[va_idx]
        y_tr_full, y_va = y[tr_idx], y[va_idx]

        # Nested split: carve an early-stopping holdout OUT OF THE TRAINING FOLD ONLY.
        # The validation fold (X_va/y_va) is never used for early stopping -> no leakage into OOF.
        X_tr, X_es, y_tr, y_es = train_test_split(
            X_tr_full, y_tr_full, test_size=EARLY_STOP_HOLDOUT_FRAC, random_state=SEED
        )

        lgb_model, xgb_model, cb_model = make_models()

        lgb_model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)],
                      callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
        xgb_model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
        cb_model.fit(X_tr, y_tr, eval_set=(X_es, y_es))

        p_lgb = lgb_model.predict(X_va)
        p_xgb = xgb_model.predict(X_va)
        p_cb = cb_model.predict(X_va)

        oof_lgb[va_idx] = p_lgb
        oof_xgb[va_idx] = p_xgb
        oof_cb[va_idx] = p_cb

        test_pred_lgb[fold] = lgb_model.predict(X_test)
        test_pred_xgb[fold] = xgb_model.predict(X_test)
        test_pred_cb[fold] = cb_model.predict(X_test)

        r2_lgb_fold = r2_score(y_va, p_lgb)
        r2_xgb_fold = r2_score(y_va, p_xgb)
        r2_cb_fold = r2_score(y_va, p_cb)
        fold_scores["lgb"].append(r2_lgb_fold)
        fold_scores["xgb"].append(r2_xgb_fold)
        fold_scores["cb"].append(r2_cb_fold)
        importances.append(lgb_model.feature_importances_)

        print(f"  Fold {fold}: LGB R2={r2_lgb_fold:.4f}  XGB R2={r2_xgb_fold:.4f}  CB R2={r2_cb_fold:.4f}")

    for name, arr in fold_scores.items():
        print(f"  {name.upper()} \u2014 mean R2={np.mean(arr):.4f}  std={np.std(arr):.4f}")

    results[tt] = {
        "lgb": {"oof_r2": r2_score(y, oof_lgb), "fold_mean": float(np.mean(fold_scores["lgb"])), "fold_std": float(np.std(fold_scores["lgb"]))},
        "xgb": {"oof_r2": r2_score(y, oof_xgb), "fold_mean": float(np.mean(fold_scores["xgb"])), "fold_std": float(np.std(fold_scores["xgb"]))},
        "cb": {"oof_r2": r2_score(y, oof_cb), "fold_mean": float(np.mean(fold_scores["cb"])), "fold_std": float(np.std(fold_scores["cb"]))},
    }
    oof_store[tt] = {"y": y, "lgb": oof_lgb, "xgb": oof_xgb, "cb": oof_cb, "tr_index": tr.index.values}
    test_pred_store[tt] = {
        "lgb": test_pred_lgb.mean(axis=0),
        "xgb": test_pred_xgb.mean(axis=0),
        "cb": test_pred_cb.mean(axis=0),
        "ids": te["id"].values if "id" in te.columns else None,
    }
    fold_importance_store[tt] = np.mean(importances, axis=0)
    print(f"\n  OOF R2 (full, out-of-fold) \u2014 LGB={results[tt]['lgb']['oof_r2']:.4f}  "
          f"XGB={results[tt]['xgb']['oof_r2']:.4f}  CB={results[tt]['cb']['oof_r2']:.4f}")


## SECTION 8 — STACKING (OOF-only meta features, fold-safe blending)

Builds a Ridge meta-learner on the three models' out-of-fold predictions (OOF-only meta-features, so no leakage), and automatically falls back to the single best base model per target if stacking does not improve on it.

In [ ]:
print("\n" + "="*70)
print("SECTION 8 — STACKING")
print("="*70)

final_test_pred = {}
stacking_used = {}

for tt in sorted(train['target_type'].unique()):
    y = oof_store[tt]["y"]
    oof_matrix = np.column_stack([oof_store[tt]["lgb"], oof_store[tt]["xgb"], oof_store[tt]["cb"]])
    best_single_r2 = max(results[tt][m]["oof_r2"] for m in ["lgb", "xgb", "cb"])

    meta = Ridge(alpha=1.0, random_state=SEED)
    meta.fit(oof_matrix, y)
    oof_stack_pred = meta.predict(oof_matrix)
    stack_r2 = r2_score(y, oof_stack_pred)

    print(f"[{tt}] Best single-model OOF R2 = {best_single_r2:.4f} | Stacked OOF R2 = {stack_r2:.4f}")

    test_matrix = np.column_stack([
        test_pred_store[tt]["lgb"], test_pred_store[tt]["xgb"], test_pred_store[tt]["cb"]
    ])

    if stack_r2 > best_single_r2:
        stacking_used[tt] = True
        final_test_pred[tt] = meta.predict(test_matrix)
        print(f"[{tt}] -> Using STACKED predictions for submission (stack beats best single model).")
    else:
        stacking_used[tt] = False
        best_model_name = max(["lgb", "xgb", "cb"], key=lambda m: results[tt][m]["oof_r2"])
        final_test_pred[tt] = test_pred_store[tt][best_model_name]
        print(f"[{tt}] -> Stacking did not beat best single model; using {best_model_name.upper()} "
              f"predictions for submission instead (avoids adding meta-learner variance for no gain).")

## SECTION 9 — DIAGNOSTICS

Feature importance (averaged across folds), residual statistics, the 5 hardest and 5 easiest training molecules by absolute residual, and a fold-stability check, computed separately for each target_type.

In [ ]:
print("\n" + "="*70)
print("SECTION 9 — DIAGNOSTICS")
print("="*70)

for tt in sorted(train['target_type'].unique()):
    print(f"\n----- Diagnostics for {tt} -----")
    tr_sub = train[train['target_type'] == tt].reset_index(drop=True)
    y = oof_store[tt]["y"]
    best_model_name = max(["lgb", "xgb", "cb"], key=lambda m: results[tt][m]["oof_r2"])
    best_oof = oof_store[tt][best_model_name]
    residuals = y - best_oof

    print(f"Best base model for diagnostics: {best_model_name.upper()}")
    print(f"Residual mean: {residuals.mean():.4f}  Residual std: {residuals.std():.4f}")

    top_feat_idx = np.argsort(fold_importance_store[tt])[::-1][:15]
    top_feats = [(feature_cols[i], float(fold_importance_store[tt][i])) for i in top_feat_idx]
    print("Top 15 features by LightGBM importance (averaged across folds):")
    for fname, imp in top_feats:
        print(f"  {fname}: {imp:.1f}")

    abs_res = np.abs(residuals)
    hardest_idx = np.argsort(abs_res)[::-1][:5]
    easiest_idx = np.argsort(abs_res)[:5]
    print("\nHardest samples (largest |residual|):")
    for i in hardest_idx:
        print(f"  smiles={tr_sub.loc[i,'smiles'][:60]}...  true={y[i]:.3f}  pred={best_oof[i]:.3f}  resid={residuals[i]:.3f}")
    print("Easiest samples (smallest |residual|):")
    for i in easiest_idx:
        print(f"  smiles={tr_sub.loc[i,'smiles'][:60]}...  true={y[i]:.3f}  pred={best_oof[i]:.3f}  resid={residuals[i]:.3f}")

    fold_std = results[tt][best_model_name]["fold_std"]
    print(f"\nFold stability ({best_model_name.upper()} R2 std across folds): {fold_std:.4f} "
          f"({'stable' if fold_std < 0.03 else 'moderate variance — investigate fold composition'})")

## SECTION 10 — SUBMISSION

Assembles `submission.csv` from both targets, applies a leakage-free exact-match override for any test molecule whose canonical SMILES matches a training molecule measured for the same target_type, validates row count/columns/id coverage against `test.csv`, and saves the file.

In [ ]:
print("\n" + "="*70)
print("SECTION 10 — SUBMISSION")
print("="*70)

# Exact-match override: if a test molecule's canonical SMILES is identical to one or more
# TRAIN molecules measured for the same target_type, use the (mean) train target value
# directly instead of the model prediction. This is NOT leakage — it only ever uses TRAIN
# labels to inform a TEST prediction for a chemically identical molecule, which is legitimate
# use of the training data. It cannot leak into the OOF/CV scores above because GroupKFold
# groups duplicate canonical SMILES together within train, so this lookup never touches
# cross-validation.
override_count_total = 0
sub_parts = []
for tt in sorted(train['target_type'].unique()):
    te_sub = test[test['target_type'] == tt].reset_index(drop=True)
    tr_sub = train[train['target_type'] == tt]
    lookup = tr_sub.groupby("canonical_smiles")["target"].mean().to_dict()

    preds = final_test_pred[tt].copy()
    override_mask = te_sub["canonical_smiles"].map(lambda s: s in lookup).values
    if override_mask.any():
        preds = np.array(preds, dtype=float)
        preds[override_mask] = te_sub.loc[override_mask, "canonical_smiles"].map(lookup).values
    n_override = int(override_mask.sum())
    override_count_total += n_override
    print(f"[{tt}] Exact train-canonical-SMILES matches in test: {n_override} / {len(te_sub)} "
          f"rows overridden with train target value.")

    part = pd.DataFrame({"id": te_sub["id"].values, "target": preds})
    sub_parts.append(part)

print(f"Total rows overridden by exact-match lookup: {override_count_total} / {len(test)}")

submission = pd.concat(sub_parts, axis=0).sort_values("id").reset_index(drop=True)

assert len(submission) == len(test), f"Row count mismatch: submission={len(submission)} test={len(test)}"
assert submission["id"].nunique() == len(submission), "Duplicate ids in submission"
assert submission.isna().sum().sum() == 0, "NaNs present in submission"
if sample_sub is not None:
    assert list(submission.columns) == list(sample_sub.columns), \
        f"Column mismatch: {list(submission.columns)} vs {list(sample_sub.columns)}"
    # NOTE: sample_submission.csv is a truncated example (10 rows) in this competition,
    # not a full id manifest, so we validate the id set against test.csv (the real manifest)
    # rather than against sample_submission.
assert set(submission["id"]) == set(test["id"]), "id set mismatch vs test.csv"

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
out_path = os.path.join(out_dir, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Saved submission -> {out_path}")
print(f"Submission shape: {submission.shape}")
print(submission.head())

print("\n" + "="*70)
print("PIPELINE COMPLETE")
print("="*70)
print(json.dumps(results, indent=2))